In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
# dbutils.widgets.text('destination_schema_zone', 'None', '07-Destination Schema Zone')   # change for testing
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')
# destination_schema_zone = dbutils.widgets.get('destination_schema_zone')    # change for testing

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_assignment_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) : # change for testing
        date_filter = f"businessdate = {run_date!r}"

        # schema =  (
        #     "AssignmentId String,"
        #     "SourceSystem String,"
        #     "PortfolioKey String,"
        #     "PortfolioName String,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId String,"
        #     "AssignmentKey String,"
        #     "CaseCode Int,"
        #     "ExposureKey String,"
        #     "PartyKey String, "
        #     "CDI String,"
        #     "TaxIdentificationNumber String,"
        #     "AgreementKey String,"
        #     "InternalAssignmentStartDate String,"
        #     "ExternalAssignmentStartDate String,"
        #     "AssignmentRecallDate timestamp,"
        #     "ChannelCode String,"
        #     "Channel String,"
        #     "ExternalCompanyAgency String,"
        #     "ExternalCompanyPartyKey String,"
        #     "ExternalCompanyFullName String,"
        #     "AssignmentTypeCode String,"
        #     "AssignmentType String,"
        #     "TeamCode String,"
        #     "Team String,"
        #     "AgentPartyKey String,"
        #     "AgentFullName String,"
        #     "AgentLoginName String,"
        #      "Supervisor String,"
        #      "AssistantManager String,"
        #      "Manager String,"
        #      "Head String,"
        #     "AssignmentBucketCode Int,"
        #     "AssignmentBucket String,"
        #     "AssignmentProcess String,"
        #     "ProcessType String,"
        #     "PacketCode Int,"
        #     "Packet String,"
        #     "PacketPreviousEndDate timestamp,"
        #     "PacketAssignmentDate timestamp,"
        #     "PaccCode Int,"
        #     "QueCode Int,"
        #     "Que String,"
        #     "Segment String,"
        #     "ExtraSegment String,"
        #     "StreamTypeCode Int,"
        #     "StreamType String,"
        #     "StreamStatusCode Int,"
        #     "StreamStatus String,"
        #     "StreamAssignedStatusCode Int,"
        #     "StreamAssignedStatus String,"
        #     "OutstandingAmount double,"
        #     "DebtAmount double,"
        #     "BusinessDate Date,"
        #    )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Assignment"   # θα γεμίσει τον πίνακα που βάζουμε εδώ # change for testing

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits", "Arotron_dec_flag"] # Legal Actions QC Decommission to Arotron - notebooks run

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        #portfolio_arotron_dec = portfolio.where(col("Arotron_dec_flag") == 1)
        #portfolio_list = [row["PortfolioName"] for row in portfolio_arotron_dec.select("PortfolioName").collect()]
         # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")


       # # ================== Delete From Delta Table ==================
       # if spark.catalog.tableExists(curated_table_path) is True:
       #     delete_from_curated_table_nj(
       #         spark,
       #         curated_table_path,
       #         run_date,  # type: ignore
       #         "arotron",
       #         portfolio_list
       #     )        
       # else:
       #     logging.info("Tables does not exist, no need to delete entries for business date!")
#
        # ================== crmbpr ==================

        crmbpr_extra_filters = None

        if crmbpr_extra_filters is not None:
            crmbpr_extra_filters.append(date_filter)
            crmbpr_final_filters = " AND ".join(crmbpr_extra_filters)
        else:
            crmbpr_final_filters = date_filter  # possible need change date

        crmbpr_selected_fields = [
            "crmBPRC_ID",
            "crmBPR_EffectiveDate",
            "crmBPR_TerminationDate",
            "crmBPR_ID",
            "crmBP_PR_ID",
            "businessdate",
        ]

        crmbpr = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmbpr",
            crmbpr_final_filters,
            crmbpr_selected_fields,
        )

        # ================== crmbprcoff ==================

        crmbprcoff_extra_filters = None

        if crmbprcoff_extra_filters is not None:
            crmbprcoff_extra_filters.append(date_filter)
            crmbprcoff_final_filters = " AND ".join(crmbprcoff_extra_filters)
        else:
            crmbprcoff_final_filters = date_filter  # possible need change date

        crmbprcoff_selected_fields = [
            "crmBPR_ID",
            "crmCOFF_ID",
            "businessdate",
        ]

        crmbprcoff = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmbprcoff",
            crmbprcoff_final_filters,
            crmbprcoff_selected_fields,
        )


        # ================== crmCOFF ==================

        crmCOFF_extra_filters = None

        if crmCOFF_extra_filters is not None:
            crmCOFF_extra_filters.append(date_filter)
            crmCOFF_final_filters = " AND ".join(crmCOFF_extra_filters)
        else:
            crmCOFF_final_filters = date_filter  # possible need change date

        crmCOFF_selected_fields = [
            "crmCOFF_CustomerCode",
            "crmCOFF_RefNo",
            "crmCOFF_ID",
            "crmoff_id",
            "businessdate",
        ]

        crmCOFF = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmCOFF",
            crmCOFF_final_filters,
            crmCOFF_selected_fields,
        )

        

        # ================== crwOwner ==================

        crwOwner_extra_filters = None

        if crwOwner_extra_filters is not None:
            crwOwner_extra_filters.append(date_filter)
            crwOwner_final_filters = " AND ".join(crwOwner_extra_filters)
        else:
            crwOwner_final_filters = date_filter  # possible need change date

        crwOwner_selected_fields = [
            "crwOwner_UserName",
            "crwOwner_Alias",
            "crwOwner_ID",
            "businessdate",
        ]

        crwOwner = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crwOwner",
            crwOwner_final_filters,
            crwOwner_selected_fields,
        )


        # ================== crmba ==================

        crmba_extra_filters = None

        if crmba_extra_filters is not None:
            crmba_extra_filters.append(date_filter)
            crmba_final_filters = " AND ".join(crmba_extra_filters)
        else:
            crmba_final_filters = date_filter  # possible need change date

        crmba_selected_fields = [
            "crmba_id",
            "crmBA_ParentBAID",
            "crmBE_ID",
            "crmBA_Status",
            "businessdate",
        ]

        crmba = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmba",
            crmba_final_filters,
            crmba_selected_fields,
        )

        

        # ================== crmpact ==================

        crmpact_extra_filters = None

        if crmpact_extra_filters is not None:
            crmpact_extra_filters.append(date_filter)
            crmpact_final_filters = " AND ".join(crmpact_extra_filters)
        else:
            crmpact_final_filters = date_filter  # possible need change date

        crmpact_selected_fields = [
            "crmPACT_PreferredDT",
            "crmPACT_ActionID",
            "crmBA_ID",
            "businessdate",
        ]

        crmpact = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmpact",
            crmpact_final_filters,
            crmpact_selected_fields,
        )


        # ================== crmiact ==================

        crmiact_extra_filters = None

        if crmiact_extra_filters is not None:
            crmiact_extra_filters.append(date_filter)
            crmiact_final_filters = " AND ".join(crmiact_extra_filters)
        else:
            crmiact_final_filters = date_filter  # possible need change date

        crmiact_selected_fields = [
            "crmIACT_DateTime",
            "crmBA_ID",
            "businessdate",
        ]

        crmiact = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmiact",
            crmiact_final_filters,
            crmiact_selected_fields,
        )

        # ================== crmBE ==================

        crmBE_extra_filters = None

        if crmBE_extra_filters is not None:
            crmBE_extra_filters.append(date_filter)
            crmBE_final_filters = " AND ".join(crmBE_extra_filters)
        else:
            crmBE_final_filters = date_filter  # possible need change date

        crmBE_selected_fields = [
            "crmBE_ID",
            "crmCOFF_ID",
            "businessdate",
        ]

        crmBE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmBE",
            crmBE_final_filters,
            crmBE_selected_fields,
        )

        # ================== crmrel ==================

        crmrel_extra_filters = None

        if crmrel_extra_filters is not None:
            crmrel_extra_filters.append(date_filter)
            crmrel_final_filters = " AND ".join(crmrel_extra_filters)
        else:
            crmrel_final_filters = date_filter  # possible need change date

        crmrel_selected_fields = [
            "crmREL_pR_ID",
            "crmREL_Interpretation",
            "crmrel_sR_ID",
            "crmREL_IsValid",
            "businessdate",
        ]

        crmrel = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmrel",
            crmrel_final_filters,
            crmrel_selected_fields,
        )


        # ==== Helper function for creating tables based on the ones we have defined above ====
        def rename_fields(df, fields, suffix):
            for field in fields:
                df = df.withColumnRenamed(field, f"{field}_{suffix}")

            renamed_fields = [f"{field}_{suffix}" for field in fields]
            return df.select(*renamed_fields)

        # =====================  Renamed copies without .alias() =====================
        ActionBA = rename_fields(crmba, crmba_selected_fields, "ActionBA")
        PhaseBA = rename_fields(crmba, crmba_selected_fields, "PhaseBA")
        PhaseBA2 = rename_fields(crmba, crmba_selected_fields, "PhaseBA2")
        PhasePact = rename_fields(crmpact, crmpact_selected_fields, "PhasePact")

        
        # =====================  Join Conditions =====================

        # ======== Join Conditions - 1st part of union ========  
        crmbpr_crmbprcoff_join_condition = (
            crmbpr["crmBPR_ID"] == crmbprcoff["crmBPR_ID"]
        )   # Ready

        crmbprcoff_crmCOFF_join_condition = (
            crmbprcoff["crmCOFF_ID"] == crmCOFF["crmCOFF_ID"]
        )   # Ready

        crmbpr_crwOwner_join_condition = (
            crmbpr["crmBP_PR_ID"] == crwOwner["crwOwner_ID"]
        )   # Ready

        # ======== Join Conditions - 2nd part of union ======== 
        
        crmba_ActionBA_crmba_PhaseBA_join_condition = (
            ActionBA["crmBA_ID_ActionBA"] == PhaseBA["crmBA_ParentBAID_PhaseBA"]
        )     # Ready

        crmba_PhaseBA_crmpact_PhasePact_join_condition = (
            PhaseBA["crmBA_ID_PhaseBA"] == PhasePact["crmBA_ID_PhasePact"]
        )     # Ready

        # --------
        filtered_crmpact = crmpact.filter(
            crmpact["crmPACT_ActionID"] == "SYS_BA_LEG_PHASE_CPL_PH_1500"
        )

        PhasePact2 = rename_fields(filtered_crmpact, crmpact_selected_fields, "PhasePact2")

        pause_statements = PhaseBA2.join(
            PhasePact2,
            on=PhasePact2["crmBA_ID_PhasePact2"] == PhaseBA2["crmBA_ID_PhaseBA2"],
            how="inner"
        ).select(
            PhaseBA2["crmba_id_PhaseBA2"],
            PhaseBA2["crmBA_ParentBAID_PhaseBA2"],
            PhasePact2["crmPACT_ActionID_PhasePact2"],
            PhasePact2["crmPACT_PreferredDT_PhasePact2"]
        )


        crmba_ActionBA_pauseStatements_join_condition = (
            ActionBA["crmba_id_ActionBA"] == pause_statements["crmBA_ParentBAID_PhaseBA2"]
        )   # Ready
        # --------

        crmba_ActionBA_crmiact_join_condition = (
            ActionBA["crmBA_ID_ActionBA"] == crmiact["crmBA_ID"] 
        )   # Ready

        crmba_PhaseBA_crmBE_join_condition = (
            PhaseBA["crmBE_ID_PhaseBA"] == crmBE["crmBE_ID"] 
        )   # Ready

        crmBE_crmCOFF_join_condition = (
            crmBE["crmCOFF_ID"] == crmCOFF["crmCOFF_ID"]
        )   # Ready

        crmba_ActionBA_crmrel_join_condition = (
            (ActionBA["crmBA_ID_ActionBA"] == crmrel["crmREL_pR_ID"]) &
            (crmrel["crmREL_Interpretation"] == "SYS_BPRC_LAYWER")
        )   # Ready

        crmrel_crwOwner_join_condition = (
            crmrel["crmrel_sR_ID"] == crwOwner["crwOwner_ID"] 
        )   # Ready

        # ============================== Joins ==============================

        # ======== Joins - 1st part of union ========
        df_joins_1st_part = crmbpr.join(
            crmbprcoff, crmbpr_crmbprcoff_join_condition, "inner"
        ).join(
            crmCOFF, crmbprcoff_crmCOFF_join_condition, "inner"
        ).join(
            crwOwner, crmbpr_crwOwner_join_condition, "inner"
        )

        df_1st_part_of_union = df_joins_1st_part.filter(
            (crmbpr["crmBPRC_ID"].isin(["BPRC_LAWYER", "BPRC_COFF_OWNER"]))
        
             & (crmbpr["crmBP_PR_ID"] != "BP_OPER_MANAG")
             & (crmCOFF["crmoff_id"].isin(["OFF_ABG_WS_CON", "OFF_ABG_WS_CUST"]))
        )


        # ======== Joins - 2nd part of union ========
        df_joins_2nd_part = ActionBA \
            .join(PhaseBA, crmba_ActionBA_crmba_PhaseBA_join_condition, "inner") \
            .join(PhasePact, crmba_PhaseBA_crmpact_PhasePact_join_condition, "inner") \
            .join(pause_statements, crmba_ActionBA_pauseStatements_join_condition, "left") \
            .join(crmiact, crmba_ActionBA_crmiact_join_condition, "left") \
            .join(crmBE, crmba_PhaseBA_crmBE_join_condition, "inner") \
            .join(crmCOFF, crmBE_crmCOFF_join_condition, "inner") \
            .join(crmrel, crmba_ActionBA_crmrel_join_condition, "inner") \
            .join(crwOwner, crmrel_crwOwner_join_condition, "inner")

        df_2nd_part_of_union = df_joins_2nd_part.filter(
            (PhasePact["crmPACT_ActionID_PhasePact"] == "SYS_BA_LEG_PHASE_CPL_PH_801") &
            (PhaseBA["crmBA_Status_PhaseBA"] == "SYS_PHASE_STATUS_2") &
            (crmrel["crmREL_IsValid"] == 1)
        )

        # =========================== Calculated fields ===========================

        # ======== Calculated fields - 1st part of union ========
        df_1st_with_calcs = df_1st_part_of_union \
            .withColumn("CustCDI", crmCOFF["crmCOFF_CustomerCode"]) \
            .withColumn("ContNum", when(crmCOFF["crmOFF_ID"] == "OFF_ABG_WS_CON", crmCOFF["crmCOFF_RefNo"]).otherwise(lit      (""))) \
            .withColumn("AsnType", when(crmbpr["crmBPRC_ID"] == "BPRC_LAWYER", lit("E")).otherwise(lit("I"))) \
            .withColumn("AsnID", crmbpr["crmbpr_id"]) \
            .withColumn("UserID", crwOwner["crwOwner_UserName"]) \
            .withColumn("AsnStartDT", crmbpr["crmBPR_EffectiveDate"]) \
            .withColumn("AsnEndDT", crmbpr["crmBPR_TerminationDate"]) \
            .withColumn("ExternalCompanyFullName", when(crmbpr["crmBPRC_ID"] == "BPRC_LAWYER", crwOwner["crwOwner_Alias"]).     otherwise(lit(None))) \
            .withColumn("AgentFullName", when(crmbpr["crmBPRC_ID"] != "BPRC_LAWYER", crwOwner["crwOwner_Alias"]).otherwise      (lit(None))) \
            .withColumn("ExportDT", current_timestamp())

        # ======== Calculated fields - 2nd part of union ========
        df_2nd_with_calcs = df_2nd_part_of_union \
            .withColumn("CustCDI", crmCOFF["crmCOFF_CustomerCode"]) \
            .withColumn("ContNum", when(crmCOFF["crmOFF_ID"] == "OFF_ABG_WS_CON", crmCOFF["crmCOFF_RefNo"]).otherwise(lit      (""))) \
            .withColumn("AsnType", lit("P")) \
            .withColumn("AsnID", PhaseBA["crmBA_ID_PhaseBA"]) \
            .withColumn("UserID", crwOwner["crwOwner_UserName"]) \
            .withColumn("AsnStartDT", PhasePact["crmPACT_PreferredDT_PhasePact"]) \
            .withColumn(
                "AsnEndDT",
                when(pause_statements["crmba_id_PhaseBA2"].isNotNull(), pause_statements["crmPACT_PreferredDT_PhasePact2"])
                .when(crmiact["crmIACT_DateTime"].isNotNull(), crmiact["crmIACT_DateTime"])
                .otherwise(lit(None))
            ) \
            .withColumn("ExternalCompanyFullName", crwOwner["crwOwner_Alias"]) \
            .withColumn("AgentFullName", lit(None)) \
            .withColumn("ExportDT", current_timestamp())


        # =============== Primary Keys (we will have a union) ===============

        # HashKeys (df_1st_part_of_union) - 1st part of union
        df_1st_part_of_union_primary_key_fields = [
            crmbpr["crmbpr_id"], # AsnID
            lit("arotron"),
        ]
        
        df_1st_part_of_union_target_table_arotron = df_1st_with_calcs.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(";", *df_1st_part_of_union_primary_key_fields),
                256,
            ),
        ).distinct()
        

        # HashKeys (df_2nd_part_of_union) - 2nd part of union
        df_2nd_part_of_union_primary_key_fields = [
            ActionBA["crmBA_ID_ActionBA"], # AsnID
            lit("arotron"),
        ]
        
        df_2nd_part_of_union_target_table_arotron = df_2nd_with_calcs.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(";", *df_2nd_part_of_union_primary_key_fields),
                256,
            ),
        ).distinct()
        

        # =============== Adding new columns ===============
        # ===== Adding new columns - 1st part of union =====
        df_1st_part_of_union_target_table_arotron = (df_1st_part_of_union_target_table_arotron.withColumn(
                "ExternalCompanyPartyKey",
                when(df_1st_part_of_union_target_table_arotron["AsnType"] == "I", None).otherwise(
                    df_1st_part_of_union_target_table_arotron["UserID"]
                ),
            ).withColumn(
                "AgentLoginName",
                when(df_1st_part_of_union_target_table_arotron["AsnType"] == "E", None).otherwise(
                    df_1st_part_of_union_target_table_arotron["UserID"]
                ),
        )
        )

        # ===== Adding new columns - 2nd part of union =====
        df_2nd_part_of_union_target_table_arotron = (df_2nd_part_of_union_target_table_arotron.withColumn(
                "ExternalCompanyPartyKey",
                when(df_2nd_part_of_union_target_table_arotron["AsnType"] == "I", None).otherwise(
                    df_2nd_part_of_union_target_table_arotron["UserID"]
                ),
            ).withColumn(
                "AgentLoginName",
                when(df_2nd_part_of_union_target_table_arotron["AsnType"] == "E", None).otherwise(
                    df_2nd_part_of_union_target_table_arotron["UserID"]
                ),
        )
        )

        # =============== Join with portfolio ===============
        # ==== Join with portfolio - 1st part of union ====
        df_1st_part_of_union_target_table_arotron_final_portfolio_join_condition = (
            substring(df_1st_part_of_union_target_table_arotron["CUSTCDI"], 1, 3) == portfolio["Prefix"]
        )

        df_1st_part_of_union_target_table_arotron = df_1st_part_of_union_target_table_arotron.join(
            portfolio,
            on=df_1st_part_of_union_target_table_arotron_final_portfolio_join_condition,
            how="left"
        )

        # ==== Join with portfolio - 2nd part of union ====
        df_2nd_part_of_union_target_table_arotron_final_portfolio_join_condition = (
            substring(df_2nd_part_of_union_target_table_arotron["CUSTCDI"], 1, 3) == portfolio["Prefix"]
        )

        df_2nd_part_of_union_target_table_arotron = df_2nd_part_of_union_target_table_arotron.join(
            portfolio,
            on=df_2nd_part_of_union_target_table_arotron_final_portfolio_join_condition,
            how="left"
        )

                                                  
        # =============== result_df_arotron_1st_part_of_union ===============
        result_df_arotron_1st_part_of_union = df_1st_part_of_union_target_table_arotron.select(
            df_1st_part_of_union_target_table_arotron.AssignmentId.alias("AssignmentId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale"))
            .cast("String")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            df_1st_part_of_union_target_table_arotron.AsnID.cast("String").alias("AssignmentKey"),
            lit(None).cast("Int").alias("CaseCode"),
            lit(None).cast("String").alias("ExposureKey"),
            df_1st_part_of_union_target_table_arotron["CUSTCDI"].cast("String").alias("PartyKey"),
            df_1st_part_of_union_target_table_arotron["CUSTCDI"].cast("String").alias("CDI"),
            lit(None).cast("String").alias("TaxIdentificationNumber"),
            df_1st_part_of_union_target_table_arotron.ContNum.alias("AgreementKey"),
            lit(None).cast("String").alias("InternalAssignmentStartDate"),
            df_1st_part_of_union_target_table_arotron.AsnStartDT.cast("String").alias("ExternalAssignmentStartDate"),
            df_1st_part_of_union_target_table_arotron.AsnEndDT.cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"), 
            df_1st_part_of_union_target_table_arotron["ExternalCompanyPartyKey"].cast("String").alias(
                "ExternalCompanyPartyKey"
            ),  
            df_1st_part_of_union_target_table_arotron.ExternalCompanyFullName.cast("String").alias(
                "ExternalCompanyFullName"
            ),  
            df_1st_part_of_union_target_table_arotron.AsnType.cast("String").alias("AssignmentTypeCode"),
            df_1st_part_of_union_target_table_arotron.AsnType.cast("String").alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"), 
            df_1st_part_of_union_target_table_arotron.AgentLoginName.cast("String").alias("AgentPartyKey"),
            df_1st_part_of_union_target_table_arotron.AgentFullName.cast("String").alias(
                "AgentFullName"
            ),  
            df_1st_part_of_union_target_table_arotron.AgentLoginName.cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            lit(None).cast("String").alias("ProcessType"),
            lit(None).cast("Int").alias("PacketCode"),
            lit(None).cast("String").alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            lit(None).cast("timestamp").alias("PacketAssignmentDate"),
            lit(None).cast("Int").alias("PaccCode"),
            lit(None).cast("Int").alias("QueCode"),
            lit(None).cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            lit(None).cast("double").alias("OutstandingAmount"),
            lit(None).cast("double").alias("DebtAmount"), 
            lit(None).cast("String").alias("CampaignName"),      
            crmbpr["businessdate"].cast("date").alias("BusinessDate"),
            #coalesce(portfolio.Arotron_dec_flag, lit("0")).alias("Arotron_dec_flag"), # Legal Actions QC Decommission to Arotron - notebooks run
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),
        ).distinct()
        

        # =============== result_df_arotron_2nd_part_of_union ===============
        result_df_arotron_2nd_part_of_union = df_2nd_part_of_union_target_table_arotron.select(
            df_2nd_part_of_union_target_table_arotron.AssignmentId.alias("AssignmentId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale"))
            .cast("String")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            df_2nd_part_of_union_target_table_arotron.AsnID.cast("String").alias("AssignmentKey"),
            lit(None).cast("Int").alias("CaseCode"),
            lit(None).cast("String").alias("ExposureKey"),
            df_2nd_part_of_union_target_table_arotron["CUSTCDI"].cast("String").alias("PartyKey"),
            df_2nd_part_of_union_target_table_arotron["CUSTCDI"].cast("String").alias("CDI"),
            lit(None).cast("String").alias("TaxIdentificationNumber"),
            df_2nd_part_of_union_target_table_arotron.ContNum.alias("AgreementKey"),
            lit(None).cast("String").alias("InternalAssignmentStartDate"),
            df_2nd_part_of_union_target_table_arotron.AsnStartDT.cast("String").alias("ExternalAssignmentStartDate"),
            df_2nd_part_of_union_target_table_arotron.AsnEndDT.cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"), 
            df_2nd_part_of_union_target_table_arotron["ExternalCompanyPartyKey"].cast("String").alias(
                "ExternalCompanyPartyKey"
            ),  
            df_2nd_part_of_union_target_table_arotron.ExternalCompanyFullName.cast("String").alias(
                "ExternalCompanyFullName"
            ),  
            df_2nd_part_of_union_target_table_arotron.AsnType.cast("String").alias("AssignmentTypeCode"),
            df_2nd_part_of_union_target_table_arotron.AsnType.cast("String").alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"), 
            df_2nd_part_of_union_target_table_arotron.AgentLoginName.cast("String").alias("AgentPartyKey"),
            df_2nd_part_of_union_target_table_arotron.AgentFullName.cast("String").alias(
                "AgentFullName"
            ),  
            df_2nd_part_of_union_target_table_arotron.AgentLoginName.cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            lit(None).cast("String").alias("ProcessType"),
            lit(None).cast("Int").alias("PacketCode"),
            lit(None).cast("String").alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            lit(None).cast("timestamp").alias("PacketAssignmentDate"),
            lit(None).cast("Int").alias("PaccCode"),
            lit(None).cast("Int").alias("QueCode"),
            lit(None).cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            lit(None).cast("double").alias("OutstandingAmount"),
            lit(None).cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),    
            ActionBA["businessdate_ActionBA"].cast("date").alias("BusinessDate"),
            #coalesce(portfolio.Arotron_dec_flag, lit("0")).alias("Arotron_dec_flag"), # Legal Actions QC Decommission to Arotron - notebooks run
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),   
        ).distinct()

        

        # ======= Union =======
        result_df_arotron = result_df_arotron_1st_part_of_union.union(
            result_df_arotron_2nd_part_of_union
        )
        
        # Legal Actions QC Decommission to Arotron - notebooks run
        # We keep only the portfolios for which the decommission of LAs has happened
        #result_df_arotron = result_df_arotron.filter("Arotron_dec_flag == 1")

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We drop the column "Arotron_dec_flag" from the final dataframe
        #result_df_arotron = result_df_arotron.drop("Arotron_dec_flag")


        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                #destination_schema_zone,    # change for testing
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_assignment_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix) # change for testing